<a href="https://colab.research.google.com/github/Carozuhe/web_scraping_agaval/blob/main/Ramirez_Dainer_Zuleta_Lizeth_Analizando_EA1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EVIDENCIA DE APRENDIZAJE 1 - WEB SCRAPING EN AGAVAL
# Método: BeautifulSoup con extracción de datos estructurados (JSON-LD) y Paginación

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time
from google.colab import files

# Lista global para consolidar los productos de todas las páginas
todos_los_productos = []

# Definimos cuántas páginas queremos descargar
paginas_a_descargar = 8

# Encabezado para simular visitas reales de navegador
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Iniciando la extracción de datos en Agaval con BeautifulSoup y paginación...\n")

for pagina in range(1, paginas_a_descargar + 1):
    # Construimos la URL agregando dinámicamente el número de página
    url_dinamica = f"https://www.agaval.com/tecnologia/computadores/portatiles?page={pagina}"
    print(f"Procesando página {pagina} de {paginas_a_descargar}... ", end="")

    try:
        response = requests.get(url_dinamica, headers=headers, timeout=10)

        # Aceptamos código 200 y 206 (respuestas válidas en e-commerce)
        if response.status_code not in [200, 206]:
            print(f"Saltada por código de error: {response.status_code}")
            continue

        soup_pag = BeautifulSoup(response.content, 'html.parser')

        # Buscamos elementos de datos estructurados del e-commerce (JSON-LD)
        scripts_json_ld = soup_pag.find_all('script', type='application/ld+json')

        conteo_pagina = 0
        for script in scripts_json_ld:
            try:
                if not script.string:
                    continue
                datos = json.loads(script.string)

                # Estructura ItemList (listados de catálogo)
                if isinstance(datos, dict) and datos.get('@type') == 'ItemList' and 'itemListElement' in datos:
                    for item_wrapper in datos['itemListElement']:
                        producto = item_wrapper.get('item', {})
                        if producto:
                            nombre = producto.get('name', 'No disponible')
                            sku = producto.get('sku', producto.get('id', 'No disponible'))
                            marca = producto.get('brand', {}).get('name', 'Agaval')

                            ofertas = producto.get('offers', {})
                            precio = 'Consultar sitio'
                            if isinstance(ofertas, dict):
                                precio_num = ofertas.get('highPrice') or ofertas.get('lowPrice') or ofertas.get('price')
                                if precio_num:
                                    precio = f"${float(precio_num):,.2f}"

                            todos_los_productos.append({
                                "Referencia (SKU)": sku,
                                "Nombre/Marca": f"[{marca}] {nombre}",
                                "Valor": precio,
                                "Características": f"Computador Portátil, Tecnología (Pág {pagina})"
                            })
                            conteo_pagina += 1

                # Estructura de Producto Individual
                elif isinstance(datos, dict) and datos.get('@type') == 'Product':
                    nombre = datos.get('name', 'No disponible')
                    sku = datos.get('sku', datos.get('id', 'No disponible'))
                    marca = datos.get('brand', {}).get('name', 'Agaval')

                    ofertas = datos.get('offers', {})
                    precio = 'Consultar sitio'
                    if isinstance(ofertas, dict):
                        precio_num = ofertas.get('price') or ofertas.get('lowPrice')
                        if precio_num:
                            precio = f"${float(precio_num):,.2f}"

                    todos_los_productos.append({
                        "Referencia (SKU)": sku,
                        "Nombre/Marca": f"[{marca}] {nombre}",
                        "Valor": precio,
                        "Características": f"Computador Portátil, Tecnología (Pág {pagina})"
                    })
                    conteo_pagina += 1

            except Exception:
                continue

        # Método de respaldo por API si el HTML de la página no incrusta la etiqueta LD+JSON directamente
        if conteo_pagina == 0:
            url_api = f"https://www.agaval.com/api/catalog_system/pub/products/search/tecnologia/computadores/portatiles?_from={(pagina-1)*12}&_to={pagina*12 - 1}"
            res_api = requests.get(url_api, headers=headers, timeout=10)
            if res_api.status_code in [200, 206]:
                for prod in res_api.json():
                    nombre = prod.get('productName', 'No disponible')
                    sku = prod.get('productReference', prod.get('productId', 'N/A'))
                    marca = prod.get('brand', 'Agaval')

                    items = prod.get('items', [])
                    precio = 'Consultar sitio'
                    if items:
                        sellers = items[0].get('sellers', [])
                        if sellers:
                            p_num = sellers[0].get('commertialOffer', {}).get('Price')
                            if p_num:
                                precio = f"${float(p_num):,.2f}"

                    todos_los_productos.append({
                        "Referencia (SKU)": sku,
                        "Nombre/Marca": f"[{marca}] {nombre}",
                        "Valor": precio,
                        "Características": f"Computador Portátil, Tecnología (Pág {pagina})"
                    })
                    conteo_pagina += 1

        print(f"¡Completada! Se encontraron {conteo_pagina} productos.")
        time.sleep(1.5)  # Respetar el servidor entre peticiones

    except Exception as e:
        print(f"Error al procesar la página {pagina}: {str(e)}")

# Creamos el DataFrame consolidador
df_paginado = pd.DataFrame(todos_los_productos)

# Eliminamos posibles registros duplicados
if not df_paginado.empty:
    df_paginado = df_paginado.drop_duplicates(subset=['Nombre/Marca'])

print(f"\nExtracción finalizada. Total de productos únicos recolectados: {len(df_paginado)}")

# Mostramos las primeras filas en pantalla
display(df_paginado.head(15))

# Exportamos y descargamos automáticamente el archivo CSV consolidado
nombre_archivo_total = "datos_agaval.csv"
df_paginado.to_csv(nombre_archivo_total, index=False, encoding='utf-8-sig', sep=';')
files.download(nombre_archivo_total)

Iniciando la extracción de datos en Agaval con BeautifulSoup y paginación...

Procesando página 1 de 8... ¡Completada! Se encontraron 50 productos.
Procesando página 2 de 8... ¡Completada! Se encontraron 46 productos.
Procesando página 3 de 8... ¡Completada! Se encontraron 50 productos.
Procesando página 4 de 8... ¡Completada! Se encontraron 44 productos.
Procesando página 5 de 8... ¡Completada! Se encontraron 48 productos.
Procesando página 6 de 8... ¡Completada! Se encontraron 12 productos.
Procesando página 7 de 8... ¡Completada! Se encontraron 48 productos.
Procesando página 8 de 8... ¡Completada! Se encontraron 36 productos.

Extracción finalizada. Total de productos únicos recolectados: 172


,Referencia (SKU),Nombre/Marca,Valor,Características
0,906477,[LENOVO] Computador Portátil Lenovo 15IRH10 In...,"$2,623,000.00","Computador Portátil, Tecnología (Pág 1)"
1,930804,[ASUS] Portátil Asus TUF FA506N Ryzen 7 7435H ...,"$3,869,900.00","Computador Portátil, Tecnología (Pág 1)"
2,910437,[APPLE] COMPUTADOR PORTATIL APPLE MACBOOK NEO ...,"$3,499,900.00","Computador Portátil, Tecnología (Pág 1)"
3,830806,[COMPUMAX] Computador Portatil Compumax Onix S...,"$1,280,475.00","Computador Portátil, Tecnología (Pág 1)"
4,810419,[LENOVO] COMPUTADOR PORTATIL LENOVO 15IRH10 I5...,"$2,199,900.00","Computador Portátil, Tecnología (Pág 1)"
5,893785,[APPLE] Apple MacBook Neo 13? 8RAM 256GB color...,"$3,733,200.00","Computador Portátil, Tecnología (Pág 1)"
6,760701,[ACER] COMPUTADOR PORTATIL ACER 390X I3/8/512/...,"$1,599,900.00","Computador Portátil, Tecnología (Pág 1)"
7,901244,[ASUS] COMPUTADOR PORTATIL ASUS 3904W I3/8/1T/...,"$1,899,900.00","Computador Portátil, Tecnología (Pág 1)"
8,885926,[APPLE] COMPUTADOR PORTATIL APPLE MACBOOK NEO ...,"$3,499,900.00","Computador Portátil, Tecnología (Pág 1)"
9,927202,[ASUS] Portátil Asus TUF Gaming FA506NCG AMD R...,"$3,518,777.00","Computador Portátil, Tecnología (Pág 1)"


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>